# Prepare RAF-DB (official, basic emotions) for SMIRK

Paste the Google Drive link of the "Basic emotion" folder sent by the RAF team. The folder is only listed (gdown cannot
download folders with more than 50 files) and original.zip, boundingbox.zip and list_patition_label.txt are downloaded
one by one. RAF-DB is for non-commercial research only and must not be redistributed: keep this notebook **Private**.

Output: `/kaggle/working/rafdb_crops/` (`images/*.png` + `meta.csv`), same layout as expw-crops.
Settings: Accelerator None, Internet on.

## 1. List the Drive folder and download the 3 needed files

In [ ]:
# Google Drive link of the "Basic emotion" folder from the RAF email (keep this notebook Private)
RAF_BASIC_LINK = ''

assert RAF_BASIC_LINK, 'paste the Google Drive link of the Basic emotion folder'
!pip install -q -U gdown
import os, gdown

# gdown cannot download folders with more than 50 files (the RAF folder also holds ~15k loose box files),
# so only list the folder and download the few files we need, one by one
import inspect
print('gdown version:', gdown.__version__)
options = dict(skip_download=True, quiet=True)
if 'remaining_ok' in inspect.signature(gdown.download_folder).parameters:  # older gdown: stop at 50 files per folder
    options['remaining_ok'] = True
listing = gdown.download_folder(RAF_BASIC_LINK, **options)
loose = [f for f in listing if f.path.endswith(('.jpg', '.txt')) and not f.path.endswith('list_patition_label.txt')]
print(f'{len(listing)} files listed ({len(loose)} loose images / text files hidden). Other files:')
for f in listing:
    if f not in loose:
        print(' ', f.id, f.path)

WANTED = ['original.zip', 'boundingbox.zip', 'list_patition_label.txt']
DOWNLOAD = '/tmp/rafdb_download'
os.makedirs(DOWNLOAD, exist_ok=True)
for name in WANTED:
    match = [f for f in listing if os.path.basename(f.path) == name]
    if not match:
        print(f'!!! {name} not found in the folder listing')
        continue
    gdown.download(id=match[0].id, output=f'{DOWNLOAD}/{name}', quiet=False)
!ls -la /tmp/rafdb_download

## 2. Extract original images, bounding boxes and labels

In [ ]:
import glob, os

DOWNLOAD = '/tmp/rafdb_download'
for name in ['original.zip', 'boundingbox.zip', 'list_patition_label.txt']:
    assert os.path.exists(f'{DOWNLOAD}/{name}') and os.path.getsize(f'{DOWNLOAD}/{name}') > 0, f'{name} was not downloaded'

!unzip -q -o {DOWNLOAD}/original.zip -d /tmp/rafdb
!unzip -q -o {DOWNLOAD}/boundingbox.zip -d /tmp/rafdb
LABEL_FILE = f'{DOWNLOAD}/list_patition_label.txt'

IMAGE_DIR = os.path.dirname(glob.glob('/tmp/rafdb/**/train_00001.jpg', recursive=True)[0])
BBOX_DIR = os.path.dirname(glob.glob('/tmp/rafdb/**/train_00001_boundingbox.txt', recursive=True)[0])
print('images:', IMAGE_DIR, len(os.listdir(IMAGE_DIR)))
print('boxes :', BBOX_DIR, len(os.listdir(BBOX_DIR)))
!head -3 {LABEL_FILE}
!cat "{BBOX_DIR}/train_00001_boundingbox.txt"

## 3. Write the crop script

In [ ]:
%%writefile /kaggle/working/prepare_rafdb.py
"""
Crop the labeled faces of the official RAF-DB basic emotion set (http://whdeng.cn/RAF/model1.html) into the same layout
as the ExpW crops (crop_expw.py).

Expected input (after extracting the official zips):
    Image/original/{train_XXXXX,test_XXXX}.jpg              original images (some contain several faces)
    Annotation/boundingbox/<image name>_boundingbox.txt     "x1 y1 x2 y2" of the labeled face
    EmoLabel/list_patition_label.txt                        "<image name>.jpg <label>", labels 1..7

RAF-DB labels are mapped to the label order used for ExpW / AffectNet:
    RAF 1 Surprise -> 5, 2 Fear -> 2, 3 Disgust -> 1, 4 Happiness -> 3, 5 Sadness -> 4, 6 Anger -> 0, 7 Neutral -> 6
The official train / test split (image name prefix) is kept.

Output:
    out_dir/images/{train_XXXXX,test_XXXX}.png   cropped faces (image_size x image_size)
    out_dir/meta.csv                             name, split, label, expression, raf_label, original box, face size
"""
import os
import csv
import argparse
import numpy as np
import cv2
from tqdm import tqdm

EXPRESSIONS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
RAF_TO_COMMON = {1: 5, 2: 2, 3: 1, 4: 3, 5: 4, 6: 0, 7: 6}


def crop_face(image, box, scale, image_size):
    # square crop centered on the face box; out-of-image regions are filled with black (same as crop_expw.py)
    left, top, right, bottom = box
    center = np.array([(left + right) / 2.0, (top + bottom) / 2.0])
    size = max(right - left, bottom - top) * scale

    s = image_size / size
    M = np.array([[s, 0, image_size / 2.0 - s * center[0]],
                  [0, s, image_size / 2.0 - s * center[1]]], dtype=np.float32)

    return cv2.warpAffine(image, M, (image_size, image_size), flags=cv2.INTER_LINEAR, borderValue=0)


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Crop the labeled faces of RAF-DB (basic emotions).')
    parser.add_argument('--image_dir', type=str, required=True, help='Folder with the original images (Image/original)')
    parser.add_argument('--bbox_dir', type=str, required=True, help='Folder with the *_boundingbox.txt files')
    parser.add_argument('--label_file', type=str, required=True, help='EmoLabel/list_patition_label.txt')
    parser.add_argument('--out_dir', type=str, required=True)
    parser.add_argument('--image_size', type=int, default=256)
    parser.add_argument('--scale', type=float, default=2.0, help='Crop size relative to the face box size')
    parser.add_argument('--min_face_size', type=float, default=160, help='Skip faces whose box is smaller than this (pixels)')
    args = parser.parse_args()

    with open(args.label_file) as f:
        labels = [line.split() for line in f if line.strip()]

    os.makedirs(os.path.join(args.out_dir, 'images'), exist_ok=True)
    rows = []
    skipped = {'too_small': 0, 'unreadable': 0, 'no_box': 0}

    for image_name, raf_label in tqdm(labels):
        stem = os.path.splitext(image_name)[0]
        box_path = os.path.join(args.bbox_dir, f'{stem}_boundingbox.txt')
        if not os.path.exists(box_path):
            skipped['no_box'] += 1
            continue
        with open(box_path) as f:
            left, top, right, bottom = [float(v) for v in f.read().split()[:4]]

        if max(right - left, bottom - top) < args.min_face_size:
            skipped['too_small'] += 1
            continue

        image = cv2.imread(os.path.join(args.image_dir, image_name))
        if image is None:
            skipped['unreadable'] += 1
            continue

        cropped = crop_face(image, (left, top, right, bottom), args.scale, args.image_size)
        cv2.imwrite(os.path.join(args.out_dir, 'images', stem + '.png'), cropped)

        label = RAF_TO_COMMON[int(raf_label)]
        rows.append([stem, stem.split('_')[0], label, EXPRESSIONS[label], int(raf_label),
                     left, top, right, bottom, max(right - left, bottom - top)])

    with open(os.path.join(args.out_dir, 'meta.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['name', 'split', 'label', 'expression', 'raf_label', 'x1', 'y1', 'x2', 'y2', 'face_size'])
        writer.writerows(rows)

    print(f'Saved {len(rows)} faces, skipped {skipped}')
    for split in ['train', 'test']:
        counts = np.bincount([r[2] for r in rows if r[1] == split], minlength=len(EXPRESSIONS))
        print(f'{split}:', dict(zip(EXPRESSIONS, counts.tolist())))

## 4. Crop the faces (faces smaller than 80 px are skipped; blurry ones are removed later by the cleaning notebook)

In [ ]:
!python /kaggle/working/prepare_rafdb.py --image_dir "{IMAGE_DIR}" --bbox_dir "{BBOX_DIR}" --label_file "{LABEL_FILE}" --out_dir /kaggle/working/rafdb_crops --min_face_size 80

## 5. Check the labels visually (each row: one expression)

In [ ]:
import pandas as pd, cv2, matplotlib.pyplot as plt

meta = pd.read_csv('/kaggle/working/rafdb_crops/meta.csv', dtype={'name': str})
print(pd.crosstab(meta['expression'], meta['split'], margins=True))

expressions = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
fig, axes = plt.subplots(7, 8, figsize=(16, 15))
for row, expr in zip(axes, expressions):
    pick = meta[meta['expression'] == expr].sample(min(8, (meta['expression'] == expr).sum()), random_state=0)
    for ax in row: ax.axis('off')
    for ax, (_, r) in zip(row, pick.iterrows()):
        ax.imshow(cv2.imread(f"/kaggle/working/rafdb_crops/images/{r['name']}.png")[..., ::-1])
    row[0].set_title(expr, loc='left', fontsize=14)
plt.tight_layout(); plt.show()

## 6. Save

*Save Version → Save & Run All*, then **Output → New Dataset** `rafdb-crops` (**Private**).